In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
#%% Imports
import os
import re
import time
import math
import sys
import random
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

# HuggingFace CLIP
from transformers import CLIPProcessor, CLIPModel

2025-10-11 15:49:12.275570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760197752.299685     222 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760197752.306743     222 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import subprocess
# Try to import open_clip (preferred). If missing, attempt to install.
try:
    import open_clip
except Exception:
    # Attempt to install open-clip (two common package sources)
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/mlfoundations/open_clip.git"]
        )
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open-clip-torch"])
    import open_clip

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Using OpenCLIP backend (MLFoundations)


In [3]:
# Use the provided utils if available for downloading images
# Kaggle dataset path: /kaggle/input/dataset/utils.py
try:
    import sys
    sys.path.append('/kaggle/input/dataset')
    from utils import download_images
    print("Loaded provided download_images from dataset/utils.py")
except Exception as e:
    print("Could not import provided download_images; will use fallback downloader.", e)
    download_images = None

Loaded provided download_images from dataset/utils.py


In [4]:
#%% Config
DATA_ROOT = Path('/kaggle/input/dataset')
TRAIN_CSV = DATA_ROOT / 'train.csv'
TEST_CSV = DATA_ROOT / 'test.csv'
SAMPLE_OUT = DATA_ROOT / 'sample_test_out.csv'

WORK_DIR = Path('/kaggle/working')
IMAGES_DIR = WORK_DIR / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# OpenCLIP model choice
OPENCLIP_MODEL = 'ViT-B-32'
OPENCLIP_PRETRAINED = 'laion2b_s34b_b79k'  # reliable LAION weights

BATCH_SIZE = 64
EMBED_DIM = 512  # CLIP base embed dim for vit-base
EPOCHS = 4
LR = 1e-4
SEED = 42

In [5]:
# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [6]:
#%% Utility functions

def clean_text(text: str) -> str:
    """Basic text cleaning for catalog_content"""
    if not isinstance(text, str):
        return ""
    # Remove multiple spaces and weird characters
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
def extract_ipq(text: str) -> Optional[float]:
    """Try to extract an Item Pack Quantity (IPQ) or quantity mention from the catalog text.
    Returns a numeric quantity if found else None.
    Examples: "Pack of 3", "2 count", "500 g", "10 pack"
    """
    if not isinstance(text, str):
        return None
    t = text.lower()
    # look for patterns like 'pack of 3', '3 pack', 'x 3', 'count 3'
    m = re.search(r'pack of\s*(\d+)', t)
    if not m:
        m = re.search(r'(\d+)\s*pack(s)?', t)
    if not m:
        m = re.search(r'(\d+)\s*count', t)
    if m:
        try:
            return float(m.group(1))
        except:
            return None
    # grams / ml multipliers (optional): extract numeric followed by g/ml/kg/l
    m2 = re.search(r'(\d+[\.,]?\d*)\s*(g|kg|ml|l|litre|litres)', t)
    if m2:
        try:
            val = float(m2.group(1).replace(',', '.'))
            unit = m2.group(2)
            # convert to a simple scalar representing size
            if unit in ['kg']:
                val *= 1000
            if unit in ['l', 'litre', 'litres']:
                val *= 1000
            return float(val)
        except:
            return None
    return None
    
        

In [8]:
#%% Data loading
print('Loading CSVs...')
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
print(f'Train rows: {len(train)}, Test rows: {len(test)}')


Loading CSVs...
Train rows: 75000, Test rows: 75000


In [32]:
train

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.890
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.120
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.970
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.340
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.490
...,...,...,...,...
74995,41424,Item Name: ICE BREAKERS Spearmint Sugar Free M...,https://m.media-amazon.com/images/I/81p9PcPsff...,10.395
74996,35537,"Item Name: Davidson's Organics, Vanilla Essenc...",https://m.media-amazon.com/images/I/51DDKoa+mb...,35.920
74997,249971,Item Name: Jolly Rancher Hard Candy - Blue Ras...,https://m.media-amazon.com/images/I/91R2XCcpUf...,50.330
74998,188322,Item Name: Nescafe Dolce Gusto Capsules - CARA...,https://m.media-amazon.com/images/I/51W40YU98+...,15.275


In [9]:
# quick sanity: ensure sample_id present
assert 'sample_id' in train.columns

# Basic preprocessing
for df in [train, test]:
    df['catalog_content'] = df['catalog_content'].fillna('').map(clean_text)
    df['ipq'] = df['catalog_content'].map(extract_ipq)

# Fill ipq with 1.0 where missing (safe default)
train['ipq'] = train['ipq'].fillna(1.0)
test['ipq'] = test['ipq'].fillna(1.0)

In [10]:
df

,sample_id,catalog_content,image_link,ipq
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,https://m.media-amazon.com/images/I/71hoAn78AW...,300.0
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,https://m.media-amazon.com/images/I/61ex8NHCIj...,1.0
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,https://m.media-amazon.com/images/I/61KCM61J8e...,1.0
3,95658,Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...,https://m.media-amazon.com/images/I/51Ex6uOH7y...,2.0
4,36806,"Item Name: McCormick Culinary Vanilla Extract,...",https://m.media-amazon.com/images/I/71QYlrOMoS...,1.0
...,...,...,...,...
74995,93616,Item Name: Good Seasons Zezty Italian Salad Dr...,https://m.media-amazon.com/images/I/51e9H27lgv...,4.0
74996,249434,"Item Name: Colombina Swirled Love Tiger Pops, ...",https://m.media-amazon.com/images/I/61IpkExmVt...,1.0
74997,162217,"Item Name: Kerns, Guava Nectar, 11.5 Fl Oz Can...",https://m.media-amazon.com/images/I/A1NMggyCLz...,1.0
74998,230487,Item Name: NY SPICE SHOP Licorice Candy - 1 Po...,https://m.media-amazon.com/images/I/81P69kEP5q...,1.0


In [11]:
train['price']

0         4.890
1        13.120
2         1.970
3        30.340
4        66.490
          ...  
74995    10.395
74996    35.920
74997    50.330
74998    15.275
74999    28.240
Name: price, Length: 75000, dtype: float64

In [12]:
# target transform: price distribution can be skewed. We'll predict log(price+1) during training
train['target'] = np.log1p(train['price'].clip(lower=0.01))


In [13]:
train['target']

0        1.773256
1        2.647592
2        1.088562
3        3.444895
4        4.211979
           ...   
74995    2.433175
74996    3.608753
74997    3.938275
74998    2.789630
74999    3.375538
Name: target, Length: 75000, dtype: float64

In [14]:
# Train/validation split
train_df, val_df = train_test_split(train, test_size=0.1, random_state=SEED)
print(f"Train split: {len(train_df)}, Val split: {len(val_df)}")

Train split: 67500, Val split: 7500


In [15]:
# If dataset already has local image files, use them. Otherwise download from image_link.

def simple_image_downloader(urls: List[str], out_dir: Path, start_idx: int = 0):
    """Fallback image downloader (serial)"""
    import requests
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    for i, url in enumerate(urls, start=start_idx):
        fname = out_dir / f'{i}.jpg'
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(fname, 'wb') as f:
                    f.write(r.content)
                saved.append(fname)
            else:
                saved.append(None)
        except Exception:
            saved.append(None)
    return saved

In [16]:
def build_image_index(df: pd.DataFrame, out_dir: Path, replace_existing: bool = False) -> dict:
    """Return a dict mapping sample_id -> local image path (or None) for rows in df."""
    mapping = {}
    # if provided download_images exists, use it
    if download_images is not None:
        print('Using provided download_images to fetch images (may retry).')
        # prepare list of (sample_id, url)
        items = list(zip(df['sample_id'].astype(str).tolist(), df['image_link'].astype(str).tolist()))
        # download_images should return mapping; adapt if necessary
        try:
            results = download_images(items, str(out_dir))
            # expected results: dict of sample_id -> local_path or ''/None
            for sid, url in items:
                local = results.get(sid, None)
                mapping[sid] = Path(local) if local else None
            return mapping
        except Exception as e:
            print('Provided download_images failed; falling back to simple downloader', e)

    # fallback: download serially
    urls = df['image_link'].fillna('').tolist()
    saved = simple_image_downloader(urls, out_dir)
    for sid, p in zip(df['sample_id'].astype(str).tolist(), saved):
        mapping[sid] = str(p) if p is not None else None
    return mapping

In [17]:
# For time reasons in notebook / competition runs, it's common to download only a subset or reuse cached images.
# We'll create separate image folders for train/val/test
train_img_dir = IMAGES_DIR / 'train'
val_img_dir = IMAGES_DIR / 'val'
test_img_dir = IMAGES_DIR / 'test'
train_img_dir.mkdir(parents=True, exist_ok=True)
val_img_dir.mkdir(parents=True, exist_ok=True)
test_img_dir.mkdir(parents=True, exist_ok=True)

In [18]:
# WARNING: full download may take long; optionally skip if images are not needed or use a cached subset.
# For completeness we show how to download for the splits (commented to avoid accidental long downloads in Kaggle preview)
# Uncomment the following lines to perform downloads.

# train_img_map = build_image_index(train_df, train_img_dir)
# val_img_map = build_image_index(val_df, val_img_dir)
# test_img_map = build_image_index(test, test_img_dir)

# For now, keep image maps empty and handle missing images in dataset by using CLIP default processing (black image)
train_img_map = {}
val_img_map = {}
test_img_map = {}

In [19]:
# Replace your old PricingDataset + collate_fn with this flexible version.

class PricingDataset(Dataset):
    def __init__(self,
                 df: pd.DataFrame,
                 image_map: dict,
                 processor: Optional[object] = None,   # HF processor or None
                 split: str = 'train'):
        """
        Returns for each sample:
          - sample_id
          - raw_image : PIL.Image
          - input_text : str
          - ipq : torch.tensor(float)
        Optionally (if processor provided):
          - image_inputs and text_inputs inside each sample (HF-format dicts)
        """
        self.df = df.reset_index(drop=True)
        self.image_map = image_map or {}
        self.processor = processor
        self.split = split

    def __len__(self):
        return len(self.df)

    def _load_image(self, sample_id: str, url: str):
        # Use local file if present
        local = self.image_map.get(str(sample_id))
        if local:
            try:
                return Image.open(local).convert('RGB')
            except Exception:
                pass
        # Try download (best-effort); else placeholder
        try:
            from io import BytesIO
            import requests
            resp = requests.get(url, timeout=6)
            return Image.open(BytesIO(resp.content)).convert('RGB')
        except Exception:
            return Image.new('RGB', (224, 224), (0, 0, 0))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sid = row['sample_id']
        text = row['catalog_content']
        ipq = float(row.get('ipq', 1.0))
        image = self._load_image(sid, row.get('image_link', ''))

        sample = {
            'sample_id': sid,
            'raw_image': image,         # PIL.Image for OpenCLIP pipeline
            'input_text': text,         # raw text for OpenCLIP pipeline
            'ipq': torch.tensor(ipq, dtype=torch.float32)
        }

        # If HF processor is provided, produce HF-style tensors as well
        if self.processor is not None:
            try:
                text_inputs = self.processor(text=[text], return_tensors='pt', padding=True, truncation=True)
                image_inputs = self.processor(images=image, return_tensors='pt')
                sample['text_inputs'] = text_inputs
                sample['image_inputs'] = image_inputs
            except Exception as e:
                # if processor fails, still return raw_image & input_text
                sample['text_inputs'] = None
                sample['image_inputs'] = None

        if self.split != 'test':
            sample['target'] = torch.tensor(float(row['target']), dtype=torch.float32)

        return sample


def collate_fn(batch):
    """
    Collate collects both raw_images+input_texts and (if present) HF-style processed tensors.
    Output keys:
      - sample_id (list)
      - raw_images (list[PIL.Image])
      - input_texts (list[str])
      - ipq (Tensor [B])
    Optional (only if dataset returned them):
      - pixel_values (Tensor [B,C,H,W])
      - input_ids (Tensor [B,seq])
      - attention_mask (Tensor [B,seq])
    - target (Tensor [B]) if present
    """
    sample_ids = [b['sample_id'] for b in batch]
    ipqs = torch.stack([b['ipq'] for b in batch])

    raw_images = [b['raw_image'] for b in batch]
    input_texts = [b['input_text'] for b in batch]

    out = {
        'sample_id': sample_ids,
        'raw_images': raw_images,
        'input_texts': input_texts,
        'ipq': ipqs
    }

    # If HF-style inputs exist and are not None, stack them
    if 'image_inputs' in batch[0] and batch[0]['image_inputs'] is not None:
        # Each b['image_inputs'] is a dict returned by HF processor: {'pixel_values': Tensor}
        pixel_values = torch.cat([b['image_inputs']['pixel_values'] for b in batch], dim=0)
        out['pixel_values'] = pixel_values

    if 'text_inputs' in batch[0] and batch[0]['text_inputs'] is not None:
        input_ids = torch.cat([b['text_inputs']['input_ids'] for b in batch], dim=0)
        attention_mask = torch.cat([b['text_inputs']['attention_mask'] for b in batch], dim=0)
        out['input_ids'] = input_ids
        out['attention_mask'] = attention_mask

    if 'target' in batch[0]:
        out['target'] = torch.stack([b['target'] for b in batch])

    return out


In [20]:
#%% Load OpenCLIP model + preprocess + tokenizer
print("Loading OpenCLIP model and transforms...")
model, _, preprocess = open_clip.create_model_and_transforms(OPENCLIP_MODEL, pretrained=OPENCLIP_PRETRAINED)
tokenizer = open_clip.get_tokenizer(OPENCLIP_MODEL)

model.to(DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print("OpenCLIP loaded:", OPENCLIP_MODEL, OPENCLIP_PRETRAINED)


Loading OpenCLIP model and transforms...
OpenCLIP loaded: ViT-B-32 laion2b_s34b_b79k


In [21]:
#%% Embedding extraction using open_clip
def extract_openclip_embeddings(dataloader: DataLoader, model, preprocess_fn, tokenizer_fn, device: torch.device):
    """
    Expects dataloader to yield dicts with keys 'raw_images' (list of PIL.Image)
    and 'input_texts' (list of str) and 'sample_id'.
    Returns: ids_list, image_embeddings_np, text_embeddings_np
    """
    img_embs = []
    txt_embs = []
    ids = []

    with torch.no_grad():
        for batch in dataloader:
            # preprocess images -> tensor batch
            raw_images = batch['raw_images']   # list of PIL.Image
            img_tensors = torch.stack([preprocess_fn(img) for img in raw_images], dim=0).to(device)

            # tokenize texts
            texts = batch['input_texts']
            text_tokens = tokenizer_fn(texts).to(device)

            image_features = model.encode_image(img_tensors)   # (B, D)
            text_features = model.encode_text(text_tokens)     # (B, D)

            # normalize
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

            img_embs.append(image_features.cpu().numpy())
            txt_embs.append(text_features.cpu().numpy())
            ids.extend(batch['sample_id'])

    img_embs = np.vstack(img_embs)
    txt_embs = np.vstack(txt_embs)
    return ids, img_embs, txt_embs

In [22]:
#%% Build small dataloaders for embedding extraction (use modest batch size)
train_dataset_small = PricingDataset(train_df, train_img_map, split='train')
val_dataset_small = PricingDataset(val_df, val_img_map, split='train')

In [23]:
import numpy as np, torch
from tqdm import tqdm

def safe_extract_openclip_embeddings(dataloader, model, preprocess_fn, tokenizer_fn, device, micro_batch_size=8):
    model.eval()
    ids_all, img_embs_list, txt_embs_list = [], [], []

    for batch in tqdm(dataloader, desc="DL batches"):
        sample_ids = batch['sample_id']
        raw_images = batch['raw_images']
        input_texts = batch['input_texts']
        n = len(raw_images)
        start = 0
        img_chunks, txt_chunks = [], []

        with torch.no_grad():
            while start < n:
                end = min(n, start + micro_batch_size)
                micro_imgs = raw_images[start:end]
                # preprocess on CPU
                img_tensors = torch.stack([preprocess_fn(im) for im in micro_imgs], dim=0).to(device)
                text_tokens = tokenizer_fn(input_texts[start:end]).to(device)

                image_features = model.encode_image(img_tensors)
                text_features = model.encode_text(text_tokens)

                image_features = image_features / (image_features.norm(p=2, dim=-1, keepdim=True) + 1e-10)
                text_features  = text_features  / (text_features.norm(p=2, dim=-1, keepdim=True) + 1e-10)

                img_chunks.append(image_features.cpu().numpy())
                txt_chunks.append(text_features.cpu().numpy())

                # free GPU memory quickly
                del img_tensors, text_tokens, image_features, text_features
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                start = end

        ids_all.extend(sample_ids)
        img_embs_list.append(np.vstack(img_chunks))
        txt_embs_list.append(np.vstack(txt_chunks))

    img_embs = np.vstack(img_embs_list) if img_embs_list else np.zeros((0, 512))
    txt_embs = np.vstack(txt_embs_list) if txt_embs_list else np.zeros((0, 512))
    return ids_all, img_embs, txt_embs


In [24]:
SAFE_DL_BATCH = 8       # top-level dataloader batch (lower → safer)
NUM_WORKERS = 2         # 0 for simpler debugging
MICRO_BATCH = 4         # move 4 images per GPU forward pass

train_loader_small = DataLoader(train_dataset_small, batch_size=SAFE_DL_BATCH,
                                shuffle=False, collate_fn=collate_fn,
                                num_workers=NUM_WORKERS, pin_memory=True)
val_loader_small = DataLoader(val_dataset_small, batch_size=SAFE_DL_BATCH,
                              shuffle=False, collate_fn=collate_fn,
                              num_workers=NUM_WORKERS, pin_memory=True)

train_ids, train_img_embs, train_txt_embs = safe_extract_openclip_embeddings(
    train_loader_small, model, preprocess, tokenizer, DEVICE, micro_batch_size=MICRO_BATCH
)
val_ids, val_img_embs, val_txt_embs = safe_extract_openclip_embeddings(
    val_loader_small, model, preprocess, tokenizer, DEVICE, micro_batch_size=MICRO_BATCH
)


DL batches: 100%|██████████| 938/938 [10:48<00:00,  1.45it/s]


In [25]:
print("Train IDs:", len(train_ids)) 
print("Train img emb shape:", train_img_embs.shape) 
print("Train txt emb shape:", train_txt_embs.shape) 
print("Example embedding (first):", train_img_embs[0][:5])

Train IDs: 67500
Train img emb shape: (67500, 512)
Train txt emb shape: (67500, 512)
Example embedding (first): [ 0.00021931 -0.1877809  -0.09004832  0.1180388  -0.0174094 ]


In [27]:
#--- Combine extracted embeddings into DataFrames --- 
train_emb_df = pd.DataFrame({'sample_id': train_ids}) 
train_emb_df['img_emb'] = list(train_img_embs) 
train_emb_df['txt_emb'] = list(train_txt_embs) 
val_emb_df = pd.DataFrame({'sample_id': val_ids}) 
val_emb_df['img_emb'] = list(val_img_embs) 
val_emb_df['txt_emb'] = list(val_txt_embs) # Merge into original dataframes 
train_df = train_df.merge(train_emb_df, on='sample_id', how='left') 
val_df = val_df.merge(val_emb_df, on='sample_id', how='left') 
print("Merged embeddings back into train_df and val_df")

Merged embeddings back into train_df and val_df


In [30]:
# --- Combine extracted embeddings into DataFrames ---
train_emb_df = pd.DataFrame({
    'sample_id': train_ids,
    'img_emb': list(train_img_embs),
    'txt_emb': list(train_txt_embs)
}).astype({'sample_id': str})

val_emb_df = pd.DataFrame({
    'sample_id': val_ids,
    'img_emb': list(val_img_embs),
    'txt_emb': list(val_txt_embs)
}).astype({'sample_id': str})

# --- Ensure sample_id consistency in main DataFrames ---
train_df['sample_id'] = train_df['sample_id'].astype(str)
val_df['sample_id'] = val_df['sample_id'].astype(str)

# --- Merge embeddings ---
train_df = train_df.merge(train_emb_df, on='sample_id', how='left')
val_df = val_df.merge(val_emb_df, on='sample_id', how='left')

print("✅ Merged embeddings back into train_df and val_df")
print("🔍 Columns now available:", train_df.columns.tolist()[:10], "...")

# === Sanity check ===
if 'img_emb' not in train_df.columns:
    raise ValueError("❌ Merge failed — 'img_emb' column missing. Check sample_id alignment.")

#%% ================== KNN ENHANCER FOR FEATURE SMOOTHING ==================

from sklearn.neighbors import NearestNeighbors
import numpy as np

class KNNEnhancer:
    def __init__(self, k: int = 8):
        self.k = k
        self.knn_img = None
        self.img_embs = None

    def fit(self, img_embs: np.ndarray):
        self.img_embs = img_embs
        self.knn_img = NearestNeighbors(n_neighbors=self.k, metric='euclidean')
        self.knn_img.fit(img_embs)
        print(f"KNNEnhancer fitted on {len(img_embs)} samples with k={self.k}")

    def enhance(self, query_img_embs: np.ndarray) -> np.ndarray:
        dists, idxs = self.knn_img.kneighbors(query_img_embs)
        enhanced = []
        for ds, ids in zip(dists, idxs):
            weights = 1.0 / (ds + 1e-6)
            weights /= weights.sum()
            neighbor_embs = self.img_embs[ids]
            enhanced_emb = (neighbor_embs * weights[:, None]).sum(axis=0)
            enhanced.append(enhanced_emb)
        return np.vstack(enhanced)

# --- Define targets before fitting ---
train_targets = train_df['target'].values
val_targets = val_df['target'].values

# --- Extract embeddings as stacked numpy arrays ---
train_img_array = np.vstack(train_df['img_emb'].values)
train_txt_array = np.vstack(train_df['txt_emb'].values)
val_img_array = np.vstack(val_df['img_emb'].values)
val_txt_array = np.vstack(val_df['txt_emb'].values)

# --- Fit and apply KNN enhancement ---
knn = KNNEnhancer(k=8)
knn.fit(train_img_array)

train_enh = knn.enhance(train_img_array)
val_enh = knn.enhance(val_img_array)

# --- Combine final features ---
train_features = np.hstack([
    train_img_array,
    train_txt_array,
    train_enh,
    train_df[['ipq']].values
])

val_features = np.hstack([
    val_img_array,
    val_txt_array,
    val_enh,
    val_df[['ipq']].values
])

print("✅ Feature shapes -- train:", train_features.shape, "val:", val_features.shape)


✅ Merged embeddings back into train_df and val_df
🔍 Columns now available: ['sample_id', 'catalog_content', 'image_link', 'price', 'ipq', 'target', 'img_emb_x', 'txt_emb_x', 'img_emb_y', 'txt_emb_y'] ...
KNNEnhancer fitted on 67500 samples with k=8
✅ Feature shapes -- train: (67500, 1537) val: (7500, 1537)


In [31]:
#%% ================== IMPROVED MLP TRAINING FOR PRICE PREDICTION ==================

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler

# ===== Normalize Features =====
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
val_features = scaler.transform(val_features)

# ===== NumpyDataset =====
class NumpyDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray = None):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return {'x': torch.from_numpy(self.X[idx])}
        return {
            'x': torch.from_numpy(self.X[idx]),
            'y': torch.tensor(self.y[idx], dtype=torch.float32)
        }

# ===== Improved MLP Model =====
class PriceRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 1024):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(-1)

# ===== Safer SMAPE Loss =====
def smape_loss_torch(y_true, y_pred):
    pred_price = torch.clamp(torch.expm1(y_pred), min=1e-3, max=1e6)
    true_price = torch.clamp(torch.expm1(y_true), min=1e-3, max=1e6)
    denom = (torch.abs(true_price) + torch.abs(pred_price)) / 2.0 + 1e-6
    loss = torch.mean(torch.abs(pred_price - true_price) / denom)
    return loss

# ===== Dataloaders =====
train_ds = NumpyDataset(train_features, train_targets)
val_ds = NumpyDataset(val_features, val_targets)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

# ===== Model, Optimizer, Scheduler =====
input_dim = train_features.shape[1]
model = PriceRegressor(input_dim=input_dim, hidden_dim=1024).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# ===== Training with Early Stopping =====
EPOCHS = 100
patience = 6
best_val = float('inf')
epochs_no_improve = 0

print(f"Training on {len(train_ds)} samples, validating on {len(val_ds)} samples...")

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    for batch in train_loader:
        x = batch['x'].to(DEVICE)
        y = batch['y'].to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = smape_loss_torch(y, pred)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    avg_train = np.mean(train_losses)

    # ---- Validation ----
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            x = batch['x'].to(DEVICE)
            y = batch['y'].to(DEVICE)
            pred = model(x)
            loss = smape_loss_torch(y, pred)
            val_losses.append(loss.item())

    avg_val = np.mean(val_losses)
    scheduler.step()

    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train SMAPE: {avg_train:.5f} | Val SMAPE: {avg_val:.5f}")

    # ---- Early stopping ----
    if avg_val < best_val:
        best_val = avg_val
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler': scaler
        }, WORK_DIR / 'best_price_mlp_v2.pth')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

print(f"Training complete. Best Val SMAPE: {best_val:.5f}")


Training on 67500 samples, validating on 7500 samples...
Epoch 001/100 | Train SMAPE: 0.60535 | Val SMAPE: 0.55048
Epoch 002/100 | Train SMAPE: 0.52611 | Val SMAPE: 0.53212
Epoch 003/100 | Train SMAPE: 0.49808 | Val SMAPE: 0.51978
Epoch 004/100 | Train SMAPE: 0.47431 | Val SMAPE: 0.51755
Epoch 005/100 | Train SMAPE: 0.45578 | Val SMAPE: 0.50789
Epoch 006/100 | Train SMAPE: 0.43968 | Val SMAPE: 0.50552
Epoch 007/100 | Train SMAPE: 0.42612 | Val SMAPE: 0.50556
Epoch 008/100 | Train SMAPE: 0.41359 | Val SMAPE: 0.51001
Epoch 009/100 | Train SMAPE: 0.40139 | Val SMAPE: 0.50544
Epoch 010/100 | Train SMAPE: 0.39147 | Val SMAPE: 0.50805
Epoch 011/100 | Train SMAPE: 0.38207 | Val SMAPE: 0.50265
Epoch 012/100 | Train SMAPE: 0.37310 | Val SMAPE: 0.49923
Epoch 013/100 | Train SMAPE: 0.36607 | Val SMAPE: 0.49671
Epoch 014/100 | Train SMAPE: 0.35806 | Val SMAPE: 0.49666
Epoch 015/100 | Train SMAPE: 0.35149 | Val SMAPE: 0.49870
Epoch 016/100 | Train SMAPE: 0.34521 | Val SMAPE: 0.50350
Epoch 017/100 |

In [39]:
#%% ================== LOAD CLIP MODEL AND PROCESSOR ==================
import torch

# Check if HuggingFace CLIP or OpenCLIP backend
try:
    import open_clip
    CLIP_BACKEND = "open_clip"
    print("✅ Using OpenCLIP backend")

    OPENCLIP_MODEL = 'ViT-B-32'
    OPENCLIP_PRETRAINED = 'laion2b_s34b_b79k'

    model, _, preprocess = open_clip.create_model_and_transforms(
        OPENCLIP_MODEL, pretrained=OPENCLIP_PRETRAINED
    )
    tokenizer = open_clip.get_tokenizer(OPENCLIP_MODEL)
    processor = preprocess  # alias for consistency
    model.to(DEVICE)
    model.eval()

except Exception:
    from transformers import CLIPProcessor, CLIPModel
    CLIP_BACKEND = "hf_clip"
    print("✅ Using HuggingFace CLIP fallback")

    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    tokenizer = processor  # alias for consistency
    model.to(DEVICE)
    model.eval()

print(f"✅ CLIP model loaded ({CLIP_BACKEND}) and ready on {DEVICE}.")

✅ Using OpenCLIP backend
✅ CLIP model loaded (open_clip) and ready on cuda.


In [40]:
#%% ================== TEST CLIP EMBEDDING EXTRACTION ==================

print("🔄 Extracting CLIP embeddings for test set...")

# Create test dataset using the same CLIP processor
test_dataset = PricingDataset(test, test_img_map, processor, split='test')
test_loader = DataLoader(
    test_dataset,
    batch_size=SAFE_DL_BATCH,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# Use the same safe extraction function as train/val
test_ids, test_img_embs, test_txt_embs = safe_extract_openclip_embeddings(
    test_loader,
    model,
    preprocess,
    tokenizer,
    DEVICE,
    micro_batch_size=MICRO_BATCH
)

print(f"✅ Extracted embeddings for {len(test_ids)} test samples.")
print("Image embedding shape:", test_img_embs.shape, "Text embedding shape:", test_txt_embs.shape)


🔄 Extracting CLIP embeddings for test set...


DL batches: 100%|██████████| 9375/9375 [1:47:17<00:00,  1.46it/s]  

✅ Extracted embeddings for 75000 test samples.
Image embedding shape: (75000, 512) Text embedding shape: (75000, 512)


In [41]:
import joblib
from pathlib import Path

# Define save path
EMB_PATH = Path(WORK_DIR) / "embeddings"
EMB_PATH.mkdir(parents=True, exist_ok=True)

# Save all embeddings
joblib.dump((train_ids, train_img_embs, train_txt_embs), EMB_PATH / "train_embs.pkl")
joblib.dump((val_ids, val_img_embs, val_txt_embs), EMB_PATH / "val_embs.pkl")
joblib.dump((test_ids, test_img_embs, test_txt_embs), EMB_PATH / "test_embs.pkl")

print("✅ Saved all embeddings to", EMB_PATH)


✅ Saved all embeddings to /kaggle/working/embeddings


In [45]:
import joblib
import os

# --- Ensure embeddings exist (load if needed) ---
if "test_img_embs" not in locals() or test_img_embs is None:
    if os.path.exists(WORK_DIR / "test_img_embs.pkl"):
        print("🔁 Reloading test embeddings from disk...")
        test_ids = joblib.load(WORK_DIR / "test_ids.pkl")
        test_img_embs = joblib.load(WORK_DIR / "test_img_embs.pkl")
        test_txt_embs = joblib.load(WORK_DIR / "test_txt_embs.pkl")
    else:
        raise ValueError("❌ Test embeddings not found in memory or disk. Please run CLIP embedding extraction first.")
else:
    print("✅ Test embeddings already in memory.")

# --- Merge embeddings ---
test_emb_df = pd.DataFrame({
    "sample_id": test_ids,
    "img_emb": list(test_img_embs),
    "txt_emb": list(test_txt_embs)
}).astype({"sample_id": str})

print("📊 Columns in test_emb_df:", test_emb_df.columns.tolist())
print("🧮 Embedding DataFrame shape:", test_emb_df.shape)

# Merge into test dataframe
test["sample_id"] = test["sample_id"].astype(str)
test = test.merge(test_emb_df, on="sample_id", how="left")

print("✅ Merged embeddings into test dataframe. Columns now include:")
print(test.columns.tolist())


✅ Test embeddings already in memory.
📊 Columns in test_emb_df: ['sample_id', 'img_emb', 'txt_emb']
🧮 Embedding DataFrame shape: (75000, 3)
✅ Merged embeddings into test dataframe. Columns now include:
['sample_id', 'catalog_content', 'image_link', 'ipq', 'img_emb_x', 'txt_emb_x', 'img_emb_y', 'txt_emb_y', 'img_emb', 'txt_emb']


In [48]:
#%% ================== TEST SET INFERENCE PIPELINE (FINAL FIXED VERSION) ==================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import torch.serialization

# --- Allow sklearn scaler to unpickle safely ---
torch.serialization.add_safe_globals([StandardScaler])

# --- Rebuild the MLP model definition (same as training) ---
class PriceRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 1024):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(-1)

# --- Load the saved MLP checkpoint ---
checkpoint_path = WORK_DIR / "best_price_mlp_v2.pth"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

# Infer input dimension dynamically
input_dim = test_img_embs.shape[1] * 3 + 1   # img_emb + txt_emb + knn_enh + ipq
mlp_model = PriceRegressor(input_dim=input_dim, hidden_dim=1024).to(DEVICE)

mlp_model.load_state_dict(checkpoint["model_state_dict"])
scaler = checkpoint["scaler"]
mlp_model.eval()

print("✅ Loaded trained PriceRegressor and scaler for inference.")

# --- Merge test embeddings ---
#test_emb_df = pd.DataFrame({
 #   "sample_id": test_ids,
  #  "img_emb": list(test_img_embs),
   # "txt_emb": list(test_txt_embs)
#}).astype({"sample_id": str})
# --- Drop old embedding columns if they exist ---
for col in ["img_emb", "txt_emb"]:
    if col in test.columns:
        test.drop(columns=col, inplace=True)
        print(f"🧹 Dropped existing column: {col}")

test["sample_id"] = test["sample_id"].astype(str)
test = test.merge(test_emb_df, on="sample_id", how="left")
print("✅ Merged embeddings into test dataframe")
print("🧩 Columns in test:", test.columns.tolist())

# --- Apply KNN enhancement using the same trained KNN model ---
test_img_array = np.vstack(test["img_emb"].values)
test_txt_array = np.vstack(test["txt_emb"].values)
test_enh = knn.enhance(test_img_array)

# --- Combine features (same structure as training) ---
test_features = np.hstack([
    test_img_array,
    test_txt_array,
    test_enh,
    test[["ipq"]].values
])

# --- Normalize with training scaler ---
test_features = scaler.transform(test_features)
test_features = test_features.astype(np.float32)

# --- Predict in batches ---
preds = []
mlp_model.eval()
with torch.no_grad():
    bs = 512
    for i in range(0, len(test_features), bs):
        x = torch.from_numpy(test_features[i:i+bs]).to(DEVICE)
        pred_log = mlp_model(x)
        pred_price = torch.expm1(pred_log).cpu().numpy()
        preds.extend(pred_price)

# --- Save submission ---
submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "price": np.clip(preds, a_min=0.01, a_max=None)
})

output_path = WORK_DIR / "sample_test_out.csv"
submission.to_csv(output_path, index=False)

print("✅ Saved final predictions to:", output_path)
print(submission.head())


✅ Loaded trained PriceRegressor and scaler for inference.
🧹 Dropped existing column: img_emb
🧹 Dropped existing column: txt_emb
✅ Merged embeddings into test dataframe
🧩 Columns in test: ['sample_id', 'catalog_content', 'image_link', 'ipq', 'img_emb_x', 'txt_emb_x', 'img_emb_y', 'txt_emb_y', 'img_emb', 'txt_emb']
✅ Saved final predictions to: /kaggle/working/sample_test_out.csv
  sample_id      price
0    100179  11.326388
1    245611  14.217560
2    146263  18.863298
3     95658   3.557815
4     36806  23.980400


In [52]:
import os

print("Files in /kaggle/working/:")
print(os.listdir("/kaggle/working"))


Files in /kaggle/working/:
['embeddings', '.virtual_documents', 'sample_test_out.csv', 'images', 'best_price_mlp_v2.pth']
